# 02 — Limpieza y transformación (NuDat 3)

**Objetivo (3.er párrafo del parcial):** identificar problemas de calidad, aplicar transformaciones justificadas (computacional + físico) y escribir datasets limpios en `data/processed/` **sin modificar** `data/raw/`.

**Enfoque:** toda la lógica vive en este notebook (no dependemos de `src/etl.py`).

**Pregunta científica (recordatorio):** $N/Z$ vs modo ($\beta^-$ / EC+$\beta^+$), correlación $Q_\beta$–$t_{1/2}$, valle BE/A y residual LDM.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

ROOT = Path("..").resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = Path(".").resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
DOCS = ROOT / "docs"

print("RAW =", RAW)
print("PROCESSED =", PROCESSED)
assert (RAW / "walletcards.csv").exists()


RAW = /home/isabel/nudat3-decay-stability/data/raw
PROCESSED = /home/isabel/nudat3-decay-stability/data/processed


## 0. Problemas detectados en la EDA (entrada)

| Problema | Dónde | Decisión |
|----------|--------|----------|
| Unidades mixtas de $t_{1/2}$ + `STABLE` | Wallet | Unificar a segundos; estables → flag + NULL |
| `Decay Modes` texto libre | Wallet | Parsear canales; dominante = mayor branching |
| Duplicados $(Z,N)$ | Chart 12 y 21 | `drop_duplicates(keep="first")` |
| Columnas vacías | Chart 21 | No usar spin/mass/Sn/Sp de ese archivo |
| Resonancias (solo $\Gamma$) | Wallet | Flag `is_resonance`; no borrar |
| $Q_{\beta^-}<0$ | Chart 12 | Conservar; filtrar en análisis |
| Abundancia escasa | Wallet | No imputar |

**Raw intacto:** no se escribe nada bajo `data/raw/`.


## 1. Cargar raw y tipar


In [2]:
wallet_raw = pd.read_csv(RAW / "walletcards.csv")
hl_raw = pd.read_csv(RAW / "nndc_nudat_data_export (10).csv")
qbe_raw = pd.read_csv(RAW / "nndc_nudat_data_export (12).csv")
pair_raw = pd.read_csv(RAW / "nndc_nudat_data_export (21).csv")

print("wallet", wallet_raw.shape, "| hl", hl_raw.shape, "| qbe", qbe_raw.shape, "| pair", pair_raw.shape)


wallet (4116, 25) | hl (3150, 3) | qbe (4106, 13) | pair (4106, 16)


## 2. Transformar Wallet

**Físico:** cada fila es un estado (base o isómero). $N=A-Z$. Energías a keV. Vidas medias a segundos cuando hay valor+unidad; `STABLE` no es un número.


In [3]:
HALF_LIFE_TO_SECONDS = {
    "ys": 1e-24, "zs": 1e-21, "as": 1e-18, "fs": 1e-15, "ps": 1e-12,
    "ns": 1e-9, "us": 1e-6, "µs": 1e-6, "ms": 1e-3,
    "s": 1.0, "m": 60.0, "h": 3600.0, "d": 86400.0,
    "y": 365.25 * 86400.0,
}

def to_float(s):
    return pd.to_numeric(s, errors="coerce")

def half_life_to_seconds(value, unit):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return np.nan
    text = str(value).strip()
    if not text or text.upper() == "STABLE":
        return np.nan
    try:
        number = float(text)
    except ValueError:
        return np.nan
    if unit is None or (isinstance(unit, float) and pd.isna(unit)):
        return np.nan
    factor = HALF_LIFE_TO_SECONDS.get(str(unit).strip().lower())
    if factor is None:
        return np.nan
    return number * factor

def level_energy_to_keV(val, unit):
    v = to_float(val)
    u = str(unit).strip().lower() if pd.notna(unit) else "kev"
    scale = {"kev": 1.0, "mev": 1e3, "ev": 1e-3}.get(u, 1.0)
    return v * scale

w = pd.DataFrame({
    "Z": to_float(wallet_raw["Atomic Number (Z)"]).astype("Int64"),
    "A": to_float(wallet_raw["Atomic Mass (A)"]).astype("Int64"),
    "level_index": to_float(wallet_raw["Level Index"]).astype("Int64"),
    "element": wallet_raw["Element"].astype(str),
    "spin_parity": wallet_raw["Spin-Parity"],
    "half_life_raw": wallet_raw["Half-Life"],
    "half_life_unit": wallet_raw["Half-Life (Unit)"],
    "abundance": to_float(wallet_raw["Abundance"]),
    "mass_excess_keV": to_float(wallet_raw["Mass Excess"]),
    "decay_modes_raw": wallet_raw["Decay Modes"],
    "decay_width": to_float(wallet_raw["Decay Width"]),
    "decay_width_unit": wallet_raw["Decay Width (Unit)"],
})
w["N"] = w["A"] - w["Z"]
w["level_energy_keV"] = [
    level_energy_to_keV(v, u)
    for v, u in zip(wallet_raw["Level Energy"], wallet_raw["Level Energy (Unit)"])
]
w["is_stable"] = w["half_life_raw"].astype(str).str.upper().eq("STABLE")
w["half_life_s_wallet"] = [
    half_life_to_seconds(v, u) for v, u in zip(w["half_life_raw"], w["half_life_unit"])
]
# Resonancia: hay ancho Γ y no hay t1/2 convertible ni STABLE
w["is_resonance"] = (
    w["decay_width"].notna()
    & ~w["is_stable"]
    & w["half_life_s_wallet"].isna()
)

print("Wallet limpio:", w.shape)
print("STABLE:", int(w["is_stable"].sum()), "| resonancias:", int(w["is_resonance"].sum()))
print("GS / isómeros:", int((w.level_index == 0).sum()), "/", int((w.level_index > 0).sum()))
w.head(3)


Wallet limpio: (4116, 17)
STABLE: 254 | resonancias: 37
GS / isómeros: 3371 / 745


,Z,A,level_index,element,spin_parity,half_life_raw,half_life_unit,abundance,mass_excess_keV,decay_modes_raw,decay_width,decay_width_unit,N,level_energy_keV,is_stable,half_life_s_wallet,is_resonance
0,0,1,0,NN,1/2+,608.9,s,NaN,8071.318060,B- = 100,NaN,NaN,1,0.0,False,608.9,False
1,1,1,0,H,1/2+,STABLE,NaN,99.9855,7288.971064,NaN,NaN,NaN,0,0.0,True,NaN,False
2,1,2,0,H,1+,STABLE,NaN,0.0145,13135.722895,NaN,NaN,NaN,1,0.0,True,NaN,False


## 3. Parsear modos de decaimiento

**Físico / computacional:**
- `B-`, `B-n`, `B-a` → clase **`B-`** (familia $\beta^-$).
- `EC`, `EC+B+`, `B+`, `ECp` → **`EC_BP`**.
- `a`/`A` → **`ALPHA`** (misma física).
- `IT` → transición isomérica.
- Resto → **`OTHER`** (detalle en `mode_code`).
- Dominante = **mayor branching**; si no hay %, el primero listado.
- `STABLE` tiene prioridad sobre el texto de modos.


In [4]:
def normalize_mode_token(token: str) -> str:
    t = token.strip().upper().replace("Α", "A").rstrip("?").strip()
    if t.startswith("B-"):
        if t.startswith("B-N") or t.startswith("B-2N") or t.startswith("B-3N"):
            return "B-n"
        if t.startswith("B-A"):
            return "B-a"
        return "B-"
    if t.startswith("EC+B+"):
        return "EC+B+"
    if t.startswith("EC"):
        if "P" in t and t != "EC":
            return "ECp"
        return "EC"
    if t.startswith("B+"):
        return "B+"
    if t in {"A", "ALPHA"} or t.startswith("A=") or t == "A":
        return "A"
    if t.startswith("IT"):
        return "IT"
    if t.startswith("N") and not t.startswith("NN"):
        return "n"
    if t.startswith("P"):
        return "p"
    if t.startswith("F"):
        return "SF"
    return t or "UNKNOWN"


def parse_decay_modes(raw) -> list:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return []
    text = str(raw).strip()
    if not text:
        return []
    channels = []
    for part in text.split(","):
        part = part.strip()
        if not part:
            continue
        m = re.match(
            r"^(.+?)\s*(?:=|~)\s*([+-]?(?:\d+\.?\d*|\.\d+)(?:[Ee][+-]?\d+)?)(?:\s+[^\s,]+)*\s*$",
            part,
        )
        if m:
            channels.append({
                "mode_code": normalize_mode_token(m.group(1)),
                "branching_pct": float(m.group(2)),
            })
        else:
            channels.append({
                "mode_code": normalize_mode_token(part),
                "branching_pct": None,
            })
    return channels


def dominant_mode_class(channels, is_stable: bool) -> str:
    if is_stable:
        return "STABLE"
    if not channels:
        return "UNKNOWN"
    # Mayor branching primero; sin % → al final (orden original estable)
    ranked = sorted(
        channels,
        key=lambda c: (c["branching_pct"] is None, -(c["branching_pct"] or 0)),
    )
    code = ranked[0]["mode_code"]
    if code in {"B-", "B-n", "B-a"}:
        return "B-"
    if code in {"EC", "EC+B+", "B+", "ECp"}:
        return "EC_BP"
    if code == "A":
        return "ALPHA"
    if code == "IT":
        return "IT"
    return "OTHER"


parsed = [parse_decay_modes(x) for x in w["decay_modes_raw"]]
w["dominant_mode"] = [
    dominant_mode_class(ch, bool(st)) for ch, st in zip(parsed, w["is_stable"])
]

# Tabla de canales (1 fila por canal)
ch_rows = []
for i, (r, channels) in enumerate(zip(w.itertuples(index=False), parsed)):
    for ch in channels:
        ch_rows.append({
            "Z": int(r.Z), "A": int(r.A), "N": int(r.N),
            "level_index": int(r.level_index),
            "mode_code": ch["mode_code"],
            "branching_pct": ch["branching_pct"],
        })
decay_channels = pd.DataFrame(ch_rows)

print("Modos dominantes (todos los estados):")
display(w["dominant_mode"].value_counts())
print("Canales parseados:", len(decay_channels))
display(decay_channels["mode_code"].value_counts().head(15))


Modos dominantes (todos los estados):


dominant_mode
B-        1545
EC_BP     1284
ALPHA      538
IT         302
STABLE     254
OTHER      193
Name: count, dtype: int64

Canales parseados: 6372


mode_code
B-       1634
EC+B+    1452
B-n      1042
A         861
IT        467
ECp       325
EC        205
SF        176
p          95
2P         18
n          17
B-a        11
2B-        10
2N          8
3N          1
Name: count, dtype: int64

## 4. Chart exports: tipar, deduplicar, descartar vacías

**Justificación:** ~747 $(Z,N)$ repetidos en (12)/(21) multiplicarían el join. Columnas vacías de (21) no aportan (spin/mass están en Wallet).


In [5]:
hl = pd.DataFrame({
    "Z": to_float(hl_raw["z"]).astype("Int64"),
    "N": to_float(hl_raw["n"]).astype("Int64"),
    "half_life_s_chart": to_float(hl_raw["halflife(Seconds)"]),
}).drop_duplicates(["Z", "N"], keep="first")

qbe = pd.DataFrame({
    "Z": to_float(qbe_raw["z"]).astype("Int64"),
    "N": to_float(qbe_raw["n"]).astype("Int64"),
    "name": qbe_raw["name"],
    "q_beta_minus_keV": to_float(qbe_raw["betaMinus"]),
    "q_ec_keV": to_float(qbe_raw["electronCapture"]),
    "q_beta_plus_keV": to_float(qbe_raw["positronEmission"]),
    "be_per_a_keV": to_float(qbe_raw["bindingEnergy"]),
    "be_ldm_residual_keV": to_float(qbe_raw["bindingEnergyLDMFit"]),
}).drop_duplicates(["Z", "N"], keep="first")

# Solo columnas con datos útiles del export 21
pair = pd.DataFrame({
    "Z": to_float(pair_raw["z"]).astype("Int64"),
    "N": to_float(pair_raw["n"]).astype("Int64"),
    "pairing_gap_keV": to_float(pair_raw["pairingGap"]),
    "q_alpha_keV": to_float(pair_raw["alpha"]),
    "delta_q_alpha_keV": to_float(pair_raw["deltaAlpha"]),
}).drop_duplicates(["Z", "N"], keep="first")

print("Tras dedup — hl:", len(hl), "qbe:", len(qbe), "pair:", len(pair))
print("Duplicados restantes qbe:", int(qbe.duplicated(["Z", "N"]).sum()))


Tras dedup — hl: 3150 qbe: 3359 pair: 3359
Duplicados restantes qbe: 0


## 5. Unificar $t_{1/2}$ y construir `nuclear_states`

**Regla:** preferir Chart (segundos) en estado base; si no, Wallet convertido. Estables → `half_life_s = NULL`. Resonancias quedan con flag.


In [6]:
states = w.merge(hl, on=["Z", "N"], how="left")

def pick_half_life(row):
    if row["is_stable"]:
        return np.nan, "none"
    # Chart solo confiable como GS para el valor de chart export
    if row["level_index"] == 0 and pd.notna(row.get("half_life_s_chart")):
        return row["half_life_s_chart"], "chart"
    if pd.notna(row["half_life_s_wallet"]):
        return row["half_life_s_wallet"], "wallet"
    return np.nan, "none"

picked = states.apply(pick_half_life, axis=1, result_type="expand")
states["half_life_s"] = picked[0]
states["half_life_source"] = picked[1]

nuclear_states = states[[
    "Z", "N", "A", "element", "level_index", "level_energy_keV", "spin_parity",
    "mass_excess_keV", "abundance", "half_life_s", "half_life_source",
    "is_stable", "is_resonance", "dominant_mode", "decay_modes_raw",
]].sort_values(["Z", "A", "level_index"]).reset_index(drop=True)

print(nuclear_states["half_life_source"].value_counts())
print("resonancias en states:", int(nuclear_states["is_resonance"].sum()))
nuclear_states.head(3)


half_life_source
chart     2897
wallet     741
none       478
Name: count, dtype: int64
resonancias en states: 37


,Z,N,A,element,level_index,level_energy_keV,spin_parity,mass_excess_keV,abundance,half_life_s,half_life_source,is_stable,is_resonance,dominant_mode,decay_modes_raw
0,0,1,1,NN,0,0.0,1/2+,8071.318060,NaN,608.9,wallet,False,False,B-,B- = 100
1,1,0,1,H,0,0.0,1/2+,7288.971064,99.9855,NaN,none,True,False,STABLE,NaN
2,1,1,2,H,0,0.0,1+,13135.722895,0.0145,NaN,none,True,False,STABLE,NaN


## 6. Join para `nuclides.csv` (solo estados base)

Tabla de análisis de la pregunta científica: GS + Q + BE/A + pairing/α.


In [7]:
gs = nuclear_states[nuclear_states["level_index"] == 0].copy()
nuclides = (
    gs.merge(qbe, on=["Z", "N"], how="left")
      .merge(pair, on=["Z", "N"], how="left")
)
nuclides["N_over_Z"] = nuclides["N"] / nuclides["Z"].replace(0, np.nan)
nuclides["name"] = nuclides["name"].fillna(
    nuclides["A"].astype(str) + nuclides["element"].astype(str)
)

nuclides = nuclides[[
    "Z", "N", "A", "element", "name", "spin_parity", "mass_excess_keV", "abundance",
    "half_life_s", "half_life_source", "is_stable", "is_resonance", "dominant_mode",
    "decay_modes_raw", "N_over_Z",
    "q_beta_minus_keV", "q_ec_keV", "q_beta_plus_keV",
    "be_per_a_keV", "be_ldm_residual_keV",
    "pairing_gap_keV", "q_alpha_keV", "delta_q_alpha_keV",
]].sort_values(["Z", "A"]).reset_index(drop=True)

print("nuclides (GS):", len(nuclides), "| dups Z,A:", int(nuclides.duplicated(["Z", "A"]).sum()))
display(nuclides["dominant_mode"].value_counts())


nuclides (GS): 3371 | dups Z,A: 0


dominant_mode
B-        1377
EC_BP     1091
ALPHA      469
STABLE     253
OTHER      181
Name: count, dtype: int64

## 7. Validación y auditoría


In [8]:
assert nuclides.duplicated(["Z", "A"]).sum() == 0, "Duplicados GS"
assert nuclear_states.duplicated(["Z", "A", "level_index"]).sum() == 0, "Duplicados estados"

audit = {
    "n_nuclides_GS": len(nuclides),
    "n_nuclear_states": len(nuclear_states),
    "n_decay_channels": len(decay_channels),
    "n_stable_GS": int(nuclides["is_stable"].sum()),
    "n_resonance_GS": int(nuclides["is_resonance"].sum()),
    "with_half_life_s": int(nuclides["half_life_s"].notna().sum()),
    "with_Qb_minus": int(nuclides["q_beta_minus_keV"].notna().sum()),
    "with_BE_A": int(nuclides["be_per_a_keV"].notna().sum()),
    "with_pairing": int(nuclides["pairing_gap_keV"].notna().sum()),
    "Qb_minus_positive": int((nuclides["q_beta_minus_keV"] > 0).sum()),
    "Qb_minus_negative": int((nuclides["q_beta_minus_keV"] < 0).sum()),
}
display(pd.Series(audit).to_frame("valor"))

print("\nOverlap join GS ∩ chart keys:")
print("  HL chart:", nuclides["half_life_source"].eq("chart").sum())
print("  BE/A:", nuclides["be_per_a_keV"].notna().mean().round(3))

# OTHER desglosado (mode_code en canales de GS)
gs_ch = decay_channels[decay_channels["level_index"] == 0]
other_gs = nuclides[nuclides["dominant_mode"] == "OTHER"][["Z", "A", "decay_modes_raw"]].head(8)
print("\nEjemplos OTHER (GS):")
display(other_gs)


,valor
n_nuclides_GS,3371
n_nuclear_states,4116
n_decay_channels,6372
n_stable_GS,253
n_resonance_GS,37
with_half_life_s,2902
with_Qb_minus,3146
with_BE_A,3358
with_pairing,3205
Qb_minus_positive,1460



Overlap join GS ∩ chart keys:
  HL chart: 2897
  BE/A: 0.996

Ejemplos OTHER (GS):


,Z,A,decay_modes_raw
4,1,4,n = 100
5,1,5,2n ~ 100
6,1,6,3n = 100
7,1,7,"2n?,4n?"
10,2,5,n = 100
12,2,7,n = 100
14,2,9,n = 100
15,2,10,2n = 100


## 8. Escribir `data/processed/` y log de limpieza

**Conservado:** todos los GS e isómeros; $Q$ negativos; BE/A = 0.
**Transformado:** unidades, parseo de modos, dedup Chart, join.
**No usado / eliminado del Chart:** filas duplicadas $(Z,N)$; columnas vacías export 21.
**No imputado:** abundancia ni vidas faltantes.


In [9]:
paths = {
    "nuclides": PROCESSED / "nuclides.csv",
    "nuclear_states": PROCESSED / "nuclear_states.csv",
    "decay_channels": PROCESSED / "decay_channels.csv",
}
nuclides.to_csv(paths["nuclides"], index=False)
nuclear_states.to_csv(paths["nuclear_states"], index=False)
decay_channels.to_csv(paths["decay_channels"], index=False)

log_lines = [
    "# Log de limpieza NuDat",
    "",
    "Generado por `notebooks/02_clean_etl.ipynb`. Raw en `data/raw/` **no modificado**.",
    "",
    "## Conteos",
    "",
    f"- nuclides (GS): {len(nuclides)}",
    f"- nuclear_states: {len(nuclear_states)}",
    f"- decay_channels: {len(decay_channels)}",
    f"- STABLE (GS): {int(nuclides['is_stable'].sum())}",
    f"- resonancias (GS): {int(nuclides['is_resonance'].sum())}",
    "",
    "## Decisiones clave",
    "",
    "- $t_{1/2}$ en segundos; Chart preferido en GS; `STABLE` → NULL + flag.",
    "- Modo dominante por mayor branching; clases B-, EC_BP, ALPHA, IT, OTHER, STABLE.",
    "- Chart deduplicado por (Z,N); columnas vacías de export 21 descartadas.",
    "- $Q<0$ y BE/A=0 conservados; filtros solo en análisis.",
    "- Isómeros en `nuclear_states`; análisis de valle/modos en GS (`nuclides`).",
    "",
    "## Modos dominantes (GS)",
    "",
]
for mode, n in nuclides["dominant_mode"].value_counts().items():
    log_lines.append(f"- `{mode}`: {n}")

log_path = DOCS / "cleaning_log.md"
log_path.write_text("\n".join(log_lines) + "\n", encoding="utf-8")

for k, p in paths.items():
    print(f"{k}: {p} ({sum(1 for _ in open(p)) - 1} data rows)")
print("log:", log_path)


nuclides: /home/isabel/nudat3-decay-stability/data/processed/nuclides.csv (3371 data rows)
nuclear_states: /home/isabel/nudat3-decay-stability/data/processed/nuclear_states.csv (4116 data rows)
decay_channels: /home/isabel/nudat3-decay-stability/data/processed/decay_channels.csv (6372 data rows)
log: /home/isabel/nudat3-decay-stability/docs/cleaning_log.md


## 9. Checklist

- [x] Raw no sobrescrito
- [x] Problemas de EDA abordados con justificación
- [x] `nuclides.csv`, `nuclear_states.csv`, `decay_channels.csv` en `data/processed/`
- [x] 0 duplicados $(Z,A)$ en nuclides
- [x] Log en `docs/cleaning_log.md`

**Siguiente etapa:** diseño relacional + MySQL / Docker.
